In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

In [13]:
download_data_dir = Path("downloaded_data/")
raw_data_dir = Path("raw_data/")
dataset_name = "wmt24_chat_dev"
devset_name = "wmt24_chat_train"

## Create tower instructions

In [14]:
from transformers import AutoTokenizer
from tower_eval.utils import write_lines, read_lines

In [15]:
t = None

# Default prompt (TInstruct)

In [16]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer, five_examples):
    examples_string = "Consider the following examples: "
    for example in five_examples:
        examples_string += (
            f"{src_lang}: {example['src']}\n{tgt_lang}: {example['ref']}\n"
        )
    return f"<|im_start|>user\n{examples_string}\nTranslate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: <|im_end|>\n<|im_start|>assistant\n"


def apply_series_for_example(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")

    return f"{final_string}"


def apply_series_of_instruction(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
    examples_string,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")

    return f"<|im_start|>user\nConsider the following examples: {examples_string}\n{final_string}<|im_end|>\n<|im_start|>assistant\n"

In [17]:
instructions_data_dir = Path("instructions/no_context_5_shot")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
    "ko": "Korean",
    "nl": "Dutch",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

for lp in lps:
    print(lp)
    if lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = lp.split("-")

    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    dev_df = pd.read_json(
        raw_data_dir / f"mt/{devset_name}.{eng}-{non_eng}/test.jsonl", lines=True
    )
    five_examples = dev_df.sample(5, random_state=42).to_dict(orient="records")
    df = pd.read_json(
        raw_data_dir / f"mt/{dataset_name}.{eng}-{non_eng}/test.jsonl", lines=True
    )
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[eng], code_lang_dict[non_eng], t, five_examples
            )
        )
        .tolist()
    )
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    dev_df = pd.read_json(
        raw_data_dir / f"mt/{devset_name}.{non_eng}-{eng}/test.jsonl", lines=True
    )
    df = pd.read_json(
        raw_data_dir / f"mt/{dataset_name}.{non_eng}-{eng}/test.jsonl", lines=True
    )
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[non_eng], code_lang_dict[eng], t, five_examples
            )
        )
        .tolist()
    )
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )

en-de
en-fr
en-pt-br
en-ko
en-nl


In [18]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer, examples_string):
    return f"<|im_start|>user\n{examples_string}\nTranslate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: <|im_end|>\n<|im_start|>assistant\n"


def apply_series_of_instruction(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
    examples_string,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")

    return f"<|im_start|>user\nConsider the following examples: {examples_string}{final_string}<|im_end|>\n<|im_start|>assistant\n"

In [19]:
np.random.seed(42)
example_strings = {}
lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"] + [
    f"{xx}-en" for xx in code_lang_dict.keys() if xx != "en"
]
for lp in lps:
    inputs_path = f"/mnt/data/jpombal/wmt24-chat-translation/instructions/full_context/mt/wmt24_chat_train.{lp}/instructions.txt"
    inputs = read_lines(inputs_path)
    refs_path = f"{raw_data_dir}/mt/wmt24_chat_train.{lp}/test.jsonl"
    refs = pd.read_json(refs_path, lines=True)["ref"].tolist()
    # make sure all examples contain context
    indeces_to_consider = [i for i in range(len(inputs)) if "Context:" in inputs[i]]
    if "nl" not in lp:
        sample_indeces = np.random.choice(indeces_to_consider, size=5)
    else:
        # there is something wrong with the Dutch dataset after N instances
        indeces_to_consider = [i for i in indeces_to_consider if i < 500]
        sample_indeces = np.random.choice(indeces_to_consider, size=5)
    example_strings[lp] = "Consider the following examples:\n"
    for i in sample_indeces:
        example_strings[lp] += f"{inputs[i]}{refs[i]}\n\n"

In [20]:
instructions_data_dir = Path("instructions/full_context_5_shot")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
    "ko": "Korean",
    "nl": "Dutch",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

for lp in lps:
    print(lp)
    if lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = lp.split("-")

    instructions = []
    df = pd.read_json(
        f"/mnt/data/jpombal/wmt24-chat-translation/raw_data/mt/{dataset_name}.{non_eng}/test.jsonl",
        lines=True,
    )
    register_convo = False
    last_i = 0
    for row_i, row in df.iterrows():
        convo_doc_id = row["doc_id"]
        if row_i != len(df) - 1:
            next_doc_id = df.loc[row_i + 1, "doc_id"]
            if convo_doc_id != next_doc_id:
                register_convo = True
                convo_df = df.loc[last_i:row_i].reset_index(drop=True)
                last_i = row_i + 1
        else:
            register_convo = True
            convo_df = df.loc[last_i:].reset_index(drop=True)
        if register_convo:
            for i, row in convo_df.iterrows():
                src_lang = code_lang_dict[row["source_language"]]
                tgt_lang = code_lang_dict[row["target_language"]]
                if (
                    row["doc_id"] == "646db6b7c49c6826f405703a"
                    and row["src"] == "Hello !"
                ):
                    a = 1
                if i == 0:
                    instructions.append(
                        apply_instruction(
                            row["src"],
                            src_lang,
                            tgt_lang,
                            t,
                            examples_string=example_strings[
                                f"{row['source_language']}-{row['target_language']}"
                            ],
                        )
                    )
                else:
                    previous_sources = convo_df.loc[: i - 1, "src"].tolist()
                    previous_directions = convo_df.loc[: i - 1, "sender"].tolist()
                    instructions.append(
                        apply_series_of_instruction(
                            row["src"],
                            src_lang,
                            tgt_lang,
                            t,
                            previous_sources=previous_sources,
                            previous_directions=previous_directions,
                            current_direction=row["sender"],
                            examples_string=example_strings[
                                f"{row['source_language']}-{row['target_language']}"
                            ],
                        )
                    )
        register_convo = False

    df["instructions"] = instructions
    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    instructions = df[df["source_language"] == "en"]["instructions"].tolist()
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    instructions = df[df["target_language"] == "en"]["instructions"].tolist()
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )

en-de
en-fr
en-pt-br
en-ko
en-nl


# Empty system llama3 prompt

In [4]:
t = AutoTokenizer.from_pretrained("Unbabel/Tower-Llama3-70B-WMT24-Merged-2106")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer, five_examples):
    examples_string = "Consider the following examples: "
    for example in five_examples:
        examples_string += (
            f"{src_lang}: {example['src']}\n{tgt_lang}: {example['ref']}\n"
        )
    return f"<|start_header_id|>system<|end_header_id|>\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{examples_string}\nTranslate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: <|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"


def apply_series_for_example(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")

    return f"{final_string}"


def apply_series_of_instruction(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
    examples_string,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")

    return f"<|start_header_id|>system<|end_header_id|>\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nConsider the following examples: {examples_string}\n{final_string}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

In [6]:
instructions_data_dir = Path("instructions/no_context_empty_sys_llama3_5_shot")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
    "ko": "Korean",
    "nl": "Dutch",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

for lp in lps:
    print(lp)
    if lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = lp.split("-")

    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    dev_df = pd.read_json(
        raw_data_dir / f"mt/{devset_name}.{eng}-{non_eng}/test.jsonl", lines=True
    )
    five_examples = dev_df.sample(5, random_state=42).to_dict(orient="records")
    df = pd.read_json(
        raw_data_dir / f"mt/{dataset_name}.{eng}-{non_eng}/test.jsonl", lines=True
    )
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[eng], code_lang_dict[non_eng], t, five_examples
            )
        )
        .tolist()
    )
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    dev_df = pd.read_json(
        raw_data_dir / f"mt/{devset_name}.{non_eng}-{eng}/test.jsonl", lines=True
    )
    df = pd.read_json(
        raw_data_dir / f"mt/{dataset_name}.{non_eng}-{eng}/test.jsonl", lines=True
    )
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[non_eng], code_lang_dict[eng], t, five_examples
            )
        )
        .tolist()
    )
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )

en-de
en-fr
en-pt-br
en-ko
en-nl


In [7]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer, examples_string):
    return f"<|start_header_id|>system<|end_header_id|>\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{examples_string}\nTranslate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: <|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"


def apply_series_of_instruction(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
    examples_string,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")

    return f"<|start_header_id|>system<|end_header_id|>\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nConsider the following examples: {examples_string}{final_string}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

In [8]:
np.random.seed(42)
example_strings = {}
lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"] + [
    f"{xx}-en" for xx in code_lang_dict.keys() if xx != "en"
]
for lp in lps:
    inputs_path = f"/mnt/data/jpombal/wmt24-chat-translation/instructions/full_context/mt/wmt24_chat_train.{lp}/instructions.txt"
    inputs = read_lines(inputs_path)
    refs_path = f"{raw_data_dir}/mt/wmt24_chat_train.{lp}/test.jsonl"
    refs = pd.read_json(refs_path, lines=True)["ref"].tolist()
    # make sure all examples contain context
    indeces_to_consider = [i for i in range(len(inputs)) if "Context:" in inputs[i]]
    if "nl" not in lp:
        sample_indeces = np.random.choice(indeces_to_consider, size=5)
    else:
        # there is something wrong with the Dutch dataset after N instances
        indeces_to_consider = [i for i in indeces_to_consider if i < 500]
        sample_indeces = np.random.choice(indeces_to_consider, size=5)
    example_strings[lp] = "Consider the following examples:\n"
    for i in sample_indeces:
        example_strings[lp] += f"{inputs[i]}{refs[i]}\n\n"

In [9]:
print(example_strings["en-ko"])

Consider the following examples:
Context: i'm been used ALPHANUMERICID,\nHello!\nMy name is NAME-F.\nThank you for contacting PRS-ORG Chat.\nMay I know what seems to be your concern today?\n전원은 들어 오는데 연결한 기기를 전혀 인식하지 못합니다.\nHello!\nIt seems like you are calling assistance for a PRS-ORG dock device.\nLet me set your expectation but your device is beyond our supported scope.\nThey are being handled by one of our dedicated team from technical department.\n\nTranslate the English source text to Korean, given the context.\nEnglish: For further assistance, please contact PHONENUMBER option 4 from 9:00 am – 9:00 pm EST (Mon – Fri) for English language or using this hotline PHONENUMBER for Korean.\nKorean: 추가적인 지원이 필요하시면 미국 동부 표준시(월~금) 오전 9시부터 오후 9시까지 PHONENUMBER(으)로 전화하여 옵션 4번(영어)으로 통화를 하거나 한국어로 대화하길 원하시는 경우 핫라인 PHONENUMBER(으)로 전화해 주세요.

Context: 관공서 제출용 시험 성적서를 받고 싶습니다.\n회사관련되 시험성적서및 테스트 서류를 다운 받을 수 있을까요?\n대한민국 입니다.\nAre you calling for the ALPHANUMERICID PRS-ORG?\n서류가 필요합니다.\n스위치는 구매하여 설치 완

In [10]:
instructions_data_dir = Path("instructions/full_context_empty_sys_llama3_5_shot")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
    "ko": "Korean",
    "nl": "Dutch",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

for lp in lps:
    print(lp)
    if lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = lp.split("-")

    instructions = []
    df = pd.read_json(
        f"/mnt/data/jpombal/wmt24-chat-translation/raw_data/mt/{dataset_name}.{non_eng}/test.jsonl",
        lines=True,
    )
    register_convo = False
    last_i = 0
    for row_i, row in df.iterrows():
        convo_doc_id = row["doc_id"]
        if row_i != len(df) - 1:
            next_doc_id = df.loc[row_i + 1, "doc_id"]
            if convo_doc_id != next_doc_id:
                register_convo = True
                convo_df = df.loc[last_i:row_i].reset_index(drop=True)
                last_i = row_i + 1
        else:
            register_convo = True
            convo_df = df.loc[last_i:].reset_index(drop=True)
        if register_convo:
            for i, row in convo_df.iterrows():
                src_lang = code_lang_dict[row["source_language"]]
                tgt_lang = code_lang_dict[row["target_language"]]
                if (
                    row["doc_id"] == "646db6b7c49c6826f405703a"
                    and row["src"] == "Hello !"
                ):
                    a = 1
                if i == 0:
                    instructions.append(
                        apply_instruction(
                            row["src"],
                            src_lang,
                            tgt_lang,
                            t,
                            examples_string=example_strings[
                                f"{row['source_language']}-{row['target_language']}"
                            ],
                        )
                    )
                else:
                    previous_sources = convo_df.loc[: i - 1, "src"].tolist()
                    previous_directions = convo_df.loc[: i - 1, "sender"].tolist()
                    instructions.append(
                        apply_series_of_instruction(
                            row["src"],
                            src_lang,
                            tgt_lang,
                            t,
                            previous_sources=previous_sources,
                            previous_directions=previous_directions,
                            current_direction=row["sender"],
                            examples_string=example_strings[
                                f"{row['source_language']}-{row['target_language']}"
                            ],
                        )
                    )
        register_convo = False

    df["instructions"] = instructions
    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    instructions = df[df["source_language"] == "en"]["instructions"].tolist()
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    instructions = df[df["target_language"] == "en"]["instructions"].tolist()
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )

en-de
en-fr
en-pt-br
en-ko
en-nl
